This notebook contains the instructions for the quality control, error correction and demultiplexing of the raw sequencing FASTQ files, R1 (barcode) and R2 (mappable sequence). The quality control is done using [`fastp`](https://github.com/OpenGene/fastp).

The conda environment (kernel) for this notebook:
```
conda create -n quality_control 
conda activate quality_control
conda install ipykernel seqkit fastp multiqc python-lmdb multiprocess zstandard
python -m ipykernel install --user --name=quality_control
```  

### Dependencies

In [ ]:
from pathlib import Path
import subprocess
import shutil
import sys

# --- Import Python Utilities ---
# -------------------------------
relative_target_path = Path("utils") / "python_utils"
project_root = Path.cwd()
found_root = None
while True:
    if (project_root / relative_target_path).is_dir():
        found_root = project_root
        break # Found it!
    # Stop if we reach the filesystem root
    if project_root == project_root.parent:
        raise FileNotFoundError(
            f"Could not find the directory structure '{relative_target_path}'"
        )
    # Go one level up for the next iteration
    project_root = project_root.parent
# Add the found project root to sys.path if it's not already there
if found_root:
    path_str = str(found_root)
    if path_str not in sys.path:
        sys.path.append(path_str)
        print(f"Added '{path_str}' to sys.path")

from utils.python_utils import (
    # Directory Paths
    workflow_dir,
    raw_reads_dir, 
    qc_reads_dir, 
    log_dir,
    # Support data
    barcode_list_file, 
    bcode_permut_dict_file,
    # Programs
    unambig_bcode_permuts, 
    r1_demultiplex_program, 
    r2_demultiplex_program,
    fastq_integrity_check,
    fastq_match_edit,
    # Functions
    execute_command,
    compression_utility,
    validate_fastq,
    concatenate_fastq_files,
    manage_directory_content_compression,
    retain_decompressed_duplicate_paths,
    extract_subseq_from_fastq,
    run_fastp,
    fetch_r_files,
    count_fq_headers,
    format_short_path,
    barcode_info_from_filename
)

!!! Hard reset: run this code block to delete all workflow output

In [ ]:
# datasets_dirpath = working_dirpath / "Tomoseq_datasets"
# qc_reads_dirpath = datasets_dirpath / "Tomoseq_reads_qc"
# report_dirpath = working_dirpath / "report_dir"
# if qc_reads_dirpath.exists():
#     shutil.rmtree(qc_reads_dirpath)
# if report_dirpath.exists():
#     shutil.rmtree(report_dirpath)

### Pre-processing: combine and validate the raw sequence files
The raw sequence files exist as separate files for each Illumina flowecell lane. The R1 files contain mostly the poly-A adapter and the barcode sequence, and the R2 files contain the actual mappable sequence. The R1 files are used just for demultiplexing. 
1. The raw files will be combined into two files for each sample, one for each read type (R1/R2).
2. The combined files will be compressed into .zstd [(Zstandard)](https://en.wikipedia.org/wiki/Zstd) format. In addition to other advantages, this compression format is much faster to read and write, reducing disk space requirements with minimal impact on processing time.
3. The combined files will be checked for corruption using `./tools/fastq_integrity_check.py` to control that every record in the file has:
    - a valid header (starts with '@')
    - a valid sequence (contains only A, C, G, T, N)
    - a valid '+' line
    - equivalent length of sequence and quality lines
    - No empty lines in the middle of the file
  
  The pipeline integrates [seqkit](https://bioinf.shenwei.me/seqkit/) for sequence file proceessing, which also has its own built-in FASTQ format validation. 


#### Format integrity and compressed storage

In [ ]:
# --- Check file integrity and store files as .zst compressed ---
# ----------------------------------------------------------------

# --- Compress the contents of the raw reads directory into .zst files ---
all_sample_dirs = raw_reads_dir.glob("*")
all_sample_dirs = [path for path in all_sample_dirs if path.is_dir()]
if not all_sample_dirs:
    raise ValueError(f"Error: No sample directories found in '{raw_reads_dir}'.")
for sample_dir in all_sample_dirs:
    compression_success = manage_directory_content_compression(
        input_dir_path=sample_dir
    )
    if not compression_success:
        print(f"Failed to manage compression of the contents of {sample_dir}")

# --- Run integrity check on the contents of the raw reads directory ---
integrity_check_log_dir = None
for sample_dir in all_sample_dirs:
    print(f"Processing {sample_dir.name}")
    sample_malformed_dir = None
    sample_integrity_log_dir = None
    try:
        # Fetch the .fastq.zst files
        input_fastq_files = sample_dir.glob("*.fastq.zst")
        input_fastq_files = [path for path in input_fastq_files if path.is_file()]
        if not input_fastq_files:
            print(f"No .fastq.zst files found in {sample_dir}")
            continue
        print(f"Found {len(input_fastq_files)} .fastq.zst files")
        for input_fastq in input_fastq_files:
            # Decompress the input file
            decompressed_fastq = input_fastq.with_suffix('')
            decompression_success = compression_utility(
                input_path=input_fastq,
                output_path=decompressed_fastq,
                compress=False
            )
            # Check for failure
            if not decompression_success \
                or not decompressed_fastq.is_file() \
                    or not decompressed_fastq.stat().st_size > 0:
                print(f"Failed to decompress {input_fastq} to {decompressed_fastq}")
                continue
            
            # Format integrity check output
            integrity_check_log_dir = Path(log_dir) / "integrity_check_logs"
            integrity_check_log_dir.mkdir(parents=True, exist_ok=True)
            # Sample-specific output
            sample_id = sample_dir.name
            sample_integrity_log_dir = integrity_check_log_dir / sample_id
            sample_integrity_log_dir.mkdir(parents=True, exist_ok=True)
            sample_malformed_dir = sample_integrity_log_dir / "malformed_reads"
            
            # Format integrity check command
            integrity_cmd = [
                'python', fastq_integrity_check,
                '--input-fastq', str(decompressed_fastq),
                '--malformed-output-dir', str(sample_malformed_dir),
                '--store-malformed',
                '--processes', '4',
                '--log-dir', str(sample_integrity_log_dir)
            ]

            # Run integrity check
            print(f"\nRunning FASTQ integrity check on {input_fastq.name}")
            exit_status = execute_command(integrity_cmd)
            if exit_status != 0:
                print(f"Integrity check failed with exit status {exit_status}")
                continue
    finally:
        # Manage compression of the contents of the sample directory
        compression_success = manage_directory_content_compression(
            input_dir_path=Path(sample_dir)
        )
        if not compression_success:
            print(f"Failed to manage compression of the contents of {sample_dir}")
        # Clean up empty directories
        if integrity_check_log_dir and integrity_check_log_dir.is_dir():
            if next(integrity_check_log_dir.iterdir(), None) is None:
                shutil.rmtree(integrity_check_log_dir)
        if sample_malformed_dir and sample_malformed_dir.is_dir():
            if next(sample_malformed_dir.iterdir(), None) is None:
                shutil.rmtree(sample_malformed_dir)

#### Combine the input files 

In [ ]:
# --- Combine files into one R1 and one R2 file per sample ---
# ------------------------------------------------------------

# Assuming every directory in raw_reads_dir is a sample prefix/name
sample_dir_list = raw_reads_dir.glob("*")
sample_dir_list = [path for path in sample_dir_list if path.is_dir()]
if not sample_dir_list:
    raise ValueError(f"Error: No sample directories found in '{raw_reads_dir}'.")

# --- Iterate through sample directories ---
for sample_dir in sample_dir_list:
    # --- Concatenate R1 files ---
    try:
        r1_pattern = 'R1'
        concatenation_success = concatenate_fastq_files(
            input_dir=sample_dir,
            target_file_pattern=r1_pattern,
            sample_prefix = sample_dir.name)
        if not concatenation_success:
            print(f"Error: Failed to concatenate R1 files for sample '{sample_dir.name}'.")
            continue
        else:
            print(f"Concatenated R1 files for sample '{sample_dir.name}'.")
    finally:
        compression_success = manage_directory_content_compression(
            input_dir_path=sample_dir
        )
        if not compression_success:
            raise ValueError(f"Error: Failed to manage compression of the contents of {sample_dir}")
    
    # --- Concatenate R2 files ---
    try:
        r2_pattern = 'R2'
        concatenation_success = concatenate_fastq_files(
            input_dir=sample_dir,
            target_file_pattern=r2_pattern,
            sample_prefix = sample_dir.name)
        if not concatenation_success:
            print(f"Error: Failed to concatenate R2 files for sample '{sample_dir.name}'.")
            continue
        else:
            print(f"Concatenated R2 files for sample '{sample_dir.name}'.")
    finally:
        compression_success = manage_directory_content_compression(
            input_dir_path=sample_dir
        )
        if not compression_success:
            raise ValueError(f"Error: Failed to manage compression of the contents of {sample_dir}")

### Extract the barcode part from the R1 file
We know that the barcode sequence is contained at positions 1:8 in R1 files. The core command is equivalent to `cat input.fq | seqkit subseq -r 1:8 > output.fq`, but it is wrapped in Python to avoid invoking the shell.

In [ ]:
# --- Call the barcode extraction function ---
# --------------------------------------------
barcode_dir = raw_reads_dir / "barcode_sequence"
barcode_dir.mkdir(parents=True, exist_ok=True)
# Assuming every directory in raw_reads_dir is a sample prefix/name
sample_dir_list = raw_reads_dir.glob("*")
sample_dir_list = [x for x in sample_dir_list if x.is_dir() and x != barcode_dir]
dir = None
for dir in sample_dir_list:
    try:
        # Ensure directory contents are neat
        management_success = manage_directory_content_compression(input_dir_path=dir)
        if not management_success:
            raise ValueError(f"Failed to manage compression of the contents of {dir}")
        
        # Fetch the R1 fastq file from the directory
        sample_prefix = dir.name
        print(f"Processing {sample_prefix}")
        print("----------------------------")
        r1_file_list = fetch_r_files(read_type="R1", input_dir=dir)
        if len(r1_file_list) == 0:
            raise ValueError(f"No R1 files found in {dir}")
        
        r1_file_list = retain_decompressed_duplicate_paths(r1_file_list)
        if len(r1_file_list) > 1:
            raise ValueError(f"Multiple R1 files found in {dir}")
        r1_file = Path(r1_file_list[0])
        print("Input file:", r1_file.name)

        # Initialize output file path
        file_prefix = r1_file.name.split(".")[0]
        barcode_fastq_output = Path(barcode_dir) / f"{file_prefix}.barcode.fastq"
        if barcode_fastq_output.is_file():
            barcode_fastq_output.unlink()

        barcode_extraction_success = extract_subseq_from_fastq(
            input_fastq=r1_file,
            output_fastq=barcode_fastq_output,
            subseq_positions=(1, 8)
        )
        if not barcode_extraction_success:
            print(f"extract_subseq_from_fastq() failed")
    finally:
        # Maintain input files in compressed state
        if dir and dir.is_dir():
            management_success = manage_directory_content_compression(input_dir_path=dir)
            if not management_success:
                print(f"Failed to manage compression of the contents of {dir}")
        if barcode_dir and barcode_dir.is_dir():
            management_success = manage_directory_content_compression(input_dir_path=barcode_dir)
            if not management_success:
                print(f"Failed to manage compression of the contents of {barcode_dir}")

### Raw read quality control filter

#### R1 quality control

In [ ]:
# --- Run fastp on R1 files ---
# -----------------------------

# Manage compression of the raw reads directory
management_success = manage_directory_content_compression(input_dir_path=raw_reads_dir)
if not management_success:
    print(f"Failed to manage compression of the contents of {raw_reads_dir}")

# Fetch R1 FASTQ files
barcode_dir = raw_reads_dir / "barcode_sequence"
fastp_log_dir = workflow_dir / "logs" / "fastp_logs"
r1_read_filepaths = fetch_r_files(read_type="R1", input_dir=barcode_dir)

# Initialize output directory
qc_reads_dir.mkdir(parents=True, exist_ok=True)

try:
    for file in r1_read_filepaths:
        print(f"Processing {file}")
        # Format output
        file_prefix = file.name.split(".")[0]
        sample_id = file_prefix.split("_")[1]
        sample_qc_dir = Path(qc_reads_dir) / sample_id
        sample_qc_dir.mkdir(parents=True, exist_ok=True)
        sample_output_fastq = Path(sample_qc_dir) / f"{file_prefix}.barcode.qc.fastq"
        sample_log_dir = Path(fastp_log_dir) / f"{sample_id}"
        if sample_output_fastq.is_file():
            sample_output_fastq.unlink()
        
        # --- Run fastp ---
        fastp_success = run_fastp(
            input_fastq=file,
            output_fastq=sample_output_fastq,
            read_type="R1",
            log_dir=sample_log_dir
        )
        if not fastp_success:
            print(f"run_fastp() failed for {file}")
finally:
    if qc_reads_dir and qc_reads_dir.is_dir():
        management_success = manage_directory_content_compression(input_dir_path=qc_reads_dir)
        if not management_success:
            print(f"Failed to manage compression of the contents of {qc_reads_dir}")
    if barcode_dir and barcode_dir.is_dir():
        management_success = manage_directory_content_compression(input_dir_path=barcode_dir)
        if not management_success:
            print(f"Failed to manage compression of the contents of {barcode_dir}")

#### R2 quality control

In [ ]:
# --- Run fastp on R2 files ---
# -----------------------------

# Manage compression of the raw reads directory
management_success = manage_directory_content_compression(input_dir_path=raw_reads_dir)
if not management_success:
    print(f"Failed to manage compression of the contents of {raw_reads_dir}")

# Fetch R2 FASTQ files
barcode_dir = raw_reads_dir / "barcode_sequence"
r2_read_filepaths = fetch_r_files(read_type="R2", input_dir=raw_reads_dir)
r2_read_filepaths = [path for path in r2_read_filepaths if path.parent != barcode_dir]

# Initialize output directory
qc_reads_dir.mkdir(parents=True, exist_ok=True)
fastp_log_dir = workflow_dir / "logs" / "fastp_logs"

try:
    for file in r2_read_filepaths:
        sample_qc_dir = None
        try:
            print(f"Processing {file}")
            # Format output
            file_prefix = file.name.split(".")[0]
            sample_id = file_prefix.split("_")[1]
            sample_qc_dir = Path(qc_reads_dir) / sample_id
            sample_qc_dir.mkdir(parents=True, exist_ok=True)
            sample_output_fastq = Path(sample_qc_dir) / f"{file_prefix}.qc.fastq"
            sample_log_dir = Path(fastp_log_dir) / f"{sample_id}"
            if sample_output_fastq.is_file():
                sample_output_fastq.unlink()
            
            # --- Run fastp ---
            fastp_success = run_fastp(
                input_fastq=file,
                output_fastq=sample_output_fastq,
                read_type="R2",
                log_dir=sample_log_dir
            )
            if not fastp_success:
                print(f"run_fastp() failed for {file}")
        finally:
            if sample_qc_dir and sample_qc_dir.is_dir():
                management_success = manage_directory_content_compression(input_dir_path=sample_qc_dir)
                if not management_success:
                    print(f"Failed to manage compression of the contents of {sample_qc_dir}")

finally:
    if qc_reads_dir and qc_reads_dir.is_dir():
        management_success = manage_directory_content_compression(input_dir_path=qc_reads_dir)
        if not management_success:
            print(f"Failed to manage compression of the contents of {qc_reads_dir}")
    if raw_reads_dir and raw_reads_dir.is_dir():
        management_success = manage_directory_content_compression(input_dir_path=raw_reads_dir)
        if not management_success:
            print(f"Failed to manage compression of the contents of {raw_reads_dir}")

#### Summary QC report from the individual `fastp` reports using `MultiQC`. 

In [ ]:
fastp_log_dir = log_dir / "fastp_logs"
if not fastp_log_dir.is_dir():
    raise FileNotFoundError(f"No directory found {fastp_log_dir}")
multiqc_dir = log_dir / "multiqc_report"
multiqc_fastp_dir = multiqc_dir / "multiqc_fastp"
multiqc_fastp_dir.mkdir(exist_ok=True, parents=True)

# --- Summary reports on R1 fastp logs ---
r1_log_list = fastp_log_dir.rglob('*R1*.log.json*')
if not r1_log_list:
    raise FileNotFoundError("No R1 json logs found")
# Copy files to temp dir
temp_dir = multiqc_dir.parent / 'R1_multiqc_temp'
if temp_dir and temp_dir.is_dir():
    shutil.rmtree(temp_dir)
temp_dir.mkdir(parents=True, exist_ok=False)
for file in r1_log_list:
    shutil.copy(file, temp_dir)

multiqc_cmd = ["multiqc", 
               "-o", multiqc_fastp_dir, 
               "--title", "'R1 Barcode Quality Control'", 
               temp_dir
               ]
exit_status = execute_command(multiqc_cmd)
if exit_status != 0:
    raise subprocess.CalledProcessError(exit_status, multiqc_cmd)
# Clean up
if temp_dir and temp_dir.is_dir():
    shutil.rmtree(temp_dir)

# --- Summary reports on R2 fastp logs ---
r1_log_list = fastp_log_dir.rglob('*R2*.log.json*')
if not r1_log_list:
    raise FileNotFoundError("No R2 json logs found")
# Copy files to temp dir
temp_dir = multiqc_dir.parent / 'R2_multiqc_temp'
if temp_dir and temp_dir.is_dir():
    shutil.rmtree(temp_dir)
temp_dir.mkdir(parents=True, exist_ok=False)
for file in r1_log_list:
    shutil.copy(file, temp_dir)

multiqc_cmd = ["multiqc", 
               "-o", multiqc_fastp_dir, 
               "--title", "'R2 Quality Control'", 
               temp_dir
               ]
exit_status = execute_command(multiqc_cmd)
if exit_status != 0:
    raise subprocess.CalledProcessError(exit_status, multiqc_cmd)
# Clean up
if temp_dir and temp_dir.is_dir():
    shutil.rmtree(temp_dir)

### Barcode error correction

#### 1. Filter out intact barcodes
Filter the QC-controlled barcode FASTQs into two subsets: ones that have an exact match among the known barcodes and ones that don't. The ones that don't will undergo error correction in the next step.

In [ ]:
# Manage compression of the raw reads directory
management_success = manage_directory_content_compression(input_dir_path=qc_reads_dir)
if not management_success:
    print(f"Failed to manage compression of the contents of {qc_reads_dir}")
# Output
processed_reads_dir = qc_reads_dir / "further_processing"
barcode_correction_dir = processed_reads_dir / "barcode_correction"
barcode_correction_dir.mkdir(parents=True, exist_ok=True)
# logging
bcode_correction_log_dir = log_dir / 'barcode_correction_logs' / 'pass_1'
bcode_correction_log_dir.mkdir(parents=True, exist_ok=True)

# Fetch R1 FASTQ files
r1_barcode_file_list = qc_reads_dir.rglob('*.barcode.qc.fastq*')
r1_barcode_file_list = [path for path in r1_barcode_file_list if validate_fastq(path)]
if not r1_barcode_file_list:
    raise FileNotFoundError(f"No R1 FASTQ files found in {qc_reads_dir}")

try:
    for file in r1_barcode_file_list:
        sample_id = file.parent.name
        print(f"Processing {file.name} of sample {sample_id}")
        
        # Format output
        sample_output_dir = barcode_correction_dir / sample_id
        sample_output_prefix = f"{sample_id}.barcode."
        
        # Run the match/edit tool
        filter_command = [
            "python3",
            fastq_match_edit,
            "--input-fastq", file,
            "--output-dir", sample_output_dir,
            "--output-prefix", sample_id,
            "--seq-filter",
            "--query-seq-file", barcode_list_file,
            "--log-dir", bcode_correction_log_dir,
            "--keep-original",
            '--header-start', '@NS',
            '--processes', '4'
        ]
        exit_status = execute_command(filter_command)
        if exit_status != 0:
            raise RuntimeError(f"Failed to run fastq_match_edit tool: {filter_command}")

finally:
    if qc_reads_dir and qc_reads_dir.is_dir():
        management_success = manage_directory_content_compression(input_dir_path=qc_reads_dir)
        if not management_success:
            print(f"Failed to manage compression of the contents of {qc_reads_dir}")
    

#### 2. Error correction 
Barcode error correction within a set Hamming distance. The corrected FASTQ records will be directly added to the intact barcode FASTQ created in the previous step. 

In [ ]:
import json

# Clean up temps
all_dir_paths = qc_reads_dir.rglob("*")
all_dir_paths = [x for x in all_dir_paths if x.is_dir()]
for dir in all_dir_paths:
    if "_temp_" in dir.name:
        print(dir.name)
        shutil.rmtree(dir)

# --- Set up logging ---
# ----------------------
import logging
import sys

def setup_barcode_correction_logging(log_file_path: Path, unique_prefix: str):
    """
    Sets up logging for the barcode correction.
    """
    # --- Configuration ---
    log_format = '%(asctime)s - %(levelname)s - %(message)s'
    log_level = logging.INFO # Set the minimum level to log (DEBUG, INFO, WARNING, ERROR, CRITICAL)
    barcode_logger = logging.getLogger(unique_prefix)
    barcode_logger.setLevel(log_level) # Set the logger's base level
    # --- Create Formatter ---
    formatter = logging.Formatter(log_format)
    # --- Create Console Handler ---
    console_handler = logging.StreamHandler(sys.stdout) # Writes to console
    console_handler.setLevel(log_level) # Set level for console output
    console_handler.setFormatter(formatter)
    barcode_logger.addHandler(console_handler)
    # --- Create File Handler ---
    try:
        # 'a' for append mode, 'w' for write mode (overwrites existing file)
        file_handler = logging.FileHandler(log_file_path, mode='w')
        file_handler.setLevel(log_level) # Set level for file output
        file_handler.setFormatter(formatter)
        barcode_logger.addHandler(file_handler)
    except IOError as e:
        raise RuntimeError(f"Failed to create log file: {e}")
    return barcode_logger

bcode_correction_log_dir = log_dir / 'barcode_correction_logs' / 'pass_2'
bcode_correction_log_dir.mkdir(parents=True, exist_ok=True)
barcode_correction_log_file = bcode_correction_log_dir.parent / f"barcode_correction_summary.log"
barcode_logger = setup_barcode_correction_logging(barcode_correction_log_file, "barcode_correction")

# Ensure all contents are maintained in a compressed format.
management_success = manage_directory_content_compression(input_dir_path=qc_reads_dir)
if not management_success:
   raise RuntimeError(f"Failed to manage compression of the contents of {qc_reads_dir.name}") 

# Define the subdirectory for the barcode correction files. This has to be present from the previous step.
processed_reads_dir = qc_reads_dir / "further_processing"
barcode_correction_dir = processed_reads_dir / "barcode_correction"
if not barcode_correction_dir.exists():
    raise FileNotFoundError(f"Barcode correction directory not found: {barcode_correction_dir}")

# --- Iterate through the directories in the barcode correction directory ---
# ---------------------------------------------------------------------------
# It is assumed that a directory corresponds to a single sample, and that a directory contains 
# the *seq_mismatch* (corrupted barcodes that are targeted for correction) and *seq_match* (intact barcode)
target_dir_list = barcode_correction_dir.glob("*")
target_dir_list = [path for path in target_dir_list if path.is_dir()]
if not target_dir_list:
    raise FileNotFoundError(f"No target directories found in {barcode_correction_dir}")

print(f"Found {len(target_dir_list)} target directories in {barcode_correction_dir.name}")

try:
    for target_dir in target_dir_list:
        print(f"--- Processing directory: {target_dir.name} ---")

        # Fetch the mismatch FASTQ files. These contain the reads that are to be corrected
        bcode_miss_file_list = list(target_dir.glob("*seq_mismatch*"))
        bcode_miss_file_list = [path for path in bcode_miss_file_list if validate_fastq(path)]
        if not bcode_miss_file_list:
            raise FileNotFoundError(f"No mismatch FASTQ files found in {target_dir}")

        bcode_match_file_list = list(target_dir.glob("*seq_match*"))
        bcode_match_file_list = [path for path in bcode_match_file_list if validate_fastq(path)]
        if not bcode_match_file_list:
            raise FileNotFoundError(f"No match FASTQ files found in {target_dir}")

        # There cannot be overlapping file paths between the match and mismatch files.
        for miss_path in bcode_miss_file_list:
            if miss_path in bcode_match_file_list:
                raise FileNotFoundError(f"Match and mismatch FASTQ files overlap: {miss_path.name}")
        
        # There needs to be an equal number of match and mismatch files.
        if len(bcode_miss_file_list) != len(bcode_match_file_list):
            raise FileNotFoundError(f"Number of mismatch FASTQ files ({len(bcode_miss_file_list)}) does not match the number of match FASTQ files ({len(bcode_match_file_list)})")
        
        # Each match file has to have a pairing mismatch file
        # that is obtainable by the file name prefix before the first period
        valid_pair_count = 0
        for miss_path in bcode_miss_file_list:
            miss_prefix = miss_path.name.split(".")[0]
            for match_path in bcode_match_file_list:
                match_prefix = match_path.name.split(".")[0]
                if match_prefix == miss_prefix:
                    valid_pair_count += 1
        if not valid_pair_count > 0:
            raise FileNotFoundError(f"No valid match/mismatch pairing found in {barcode_correction_dir.name}")
        elif valid_pair_count != len(bcode_match_file_list):
            raise FileNotFoundError(f"Number of mismatch FASTQ files ({len(bcode_miss_file_list)}) does not match the number of match FASTQ files ({len(bcode_match_file_list)})")
        print(f"{valid_pair_count} match/mismatch pairs")
                    

        # --- Iterate through the mismatch FASTQ files in the target directory ---
        # ------------------------------------------------------------------------
        for miss_path in bcode_miss_file_list:
            dynamic_miss_file = None
            dynamic_corrected_file = None
            try:
                # --- Define the corresponding intact (mismatch) barcode file ---
                miss_prefix = miss_path.name
                miss_prefix = miss_prefix.split(".")[0]
                
                # Find the corresponding intact barcode file. the corrected barcodes will be added to this file.
                intact_barcode_file = []
                for match_path in bcode_match_file_list:
                    match_prefix = match_path.name
                    match_prefix = match_prefix.split(".")[0]
                    if match_prefix == miss_prefix:
                        intact_barcode_file.append(match_path)
                # Check for anomalies
                if not intact_barcode_file:
                    raise FileNotFoundError(f"No corresponding intact barcode file found for {miss_path.name}")
                elif len(intact_barcode_file) > 1:
                    raise RuntimeError(f"Multiple corresponding intact barcode files found for {miss_path.name}: {intact_barcode_file}")
                else:
                    intact_barcode_file = intact_barcode_file[0]

                # --- Decompress the input files if needed ---
                # --------------------------------------------
                if miss_path.suffix in ['.gz','.zst']:
                    decompressed_fastq = miss_path.with_suffix('')
                    if decompressed_fastq.is_file():
                        decompressed_fastq.unlink()
                    decompression_success = compression_utility(
                        input_path=miss_path,
                        output_path=decompressed_fastq,
                        compress=False
                    )
                    if not decompression_success \
                        or not decompressed_fastq.is_file() \
                            or not decompressed_fastq.stat().st_size > 0:
                        raise RuntimeError(f"Failed to decompress {miss_path} to {decompressed_fastq}")
                    miss_path = decompressed_fastq

                if intact_barcode_file.suffix in ['.gz','.zst']:
                    decompressed_fastq = intact_barcode_file.with_suffix('')
                    if decompressed_fastq.is_file():
                        decompressed_fastq.unlink()
                    decompression_success = compression_utility(
                        input_path=intact_barcode_file,
                        output_path=decompressed_fastq,
                        compress=False
                    )
                    if not decompression_success \
                        or not decompressed_fastq.is_file() \
                            or not decompressed_fastq.stat().st_size > 0:
                        raise RuntimeError(f"Failed to decompress {intact_barcode_file} to {decompressed_fastq}")
                    intact_barcode_file = decompressed_fastq

                
                # --- Initialize the dynamic intermediate files ---
                # This file will be updated to iteratively remove the corrected records from subsequent iterations
                dynamic_miss_file = Path(target_dir / f"{miss_prefix}_dynamic_miss.fastq")
                if dynamic_miss_file.is_file():
                    dynamic_miss_file.unlink()
                    
                # This file will be used to dynamically store the corrected records, and then append to the final output file with correct barcodes.
                dynamic_corrected_file = Path(target_dir / f"{miss_prefix}_dynamic_corrected.fastq")
                if dynamic_corrected_file.is_file():
                    dynamic_corrected_file.unlink()
                    
                # initialize the dynamic mismatch file by copying the target (corrupt) file contents into it.
                # This file will get progressively smaller as more records are corrected from it. 
                with open(miss_path, 'rb') as f_in:
                    with open(dynamic_miss_file, 'wb') as f_out:
                        shutil.copyfileobj(f_in, f_out)
                if not dynamic_miss_file.is_file() or not dynamic_miss_file.stat().st_size > 0:
                    raise RuntimeError(f"Failed to create dynamic mismatch file {dynamic_miss_file}")
                    
                # --- Create barcode permutation dictionary if needed ---
                # -------------------------------------------------------
                if not bcode_permut_dict_file.is_file():
                    print(f"Dictionary file {str(bcode_permut_dict_file)} not found. Creating it ...")
                    bcode_permut_cmd = ["python", unambig_bcode_permuts, "--maxHamming", str(2)]
                    exit_status = execute_command(bcode_permut_cmd)
                    if exit_status != 0:
                        raise subprocess.CalledProcessError(exit_status, bcode_permut_cmd)
                    print(f"Created {str(bcode_permut_dict_file)}")
                else:
                    print(f"Using existing barcode dictionary file: {str(bcode_permut_dict_file.name)}")

                # Load the barcode dictionary
                with open(bcode_permut_dict_file, 'r') as f:
                    barcode_dict = json.load(f)

                # --- Define logging output for the individual barcode log files ---
                # ------------------------------------------------------------------
                # Create a log directory per sample
                sample_log_dir = bcode_correction_log_dir / f"{miss_prefix}_barcode_correction"
                if sample_log_dir.is_dir():
                    shutil.rmtree(sample_log_dir)
                sample_log_dir.mkdir(parents=True, exist_ok=False)
                
                # --- Iterate through the original barcodes ---
                # --------------------------------------------- 
                first_line = True
                dynamic_temp_dir = None
                permutation_file = None
                try:
                    with open(barcode_list_file, 'r') as barcode_list:
                        # iterate thrrough the barcode list
                        for line in barcode_list:
                            # skipping header
                            if first_line:
                                first_line = False
                                continue
                            
                            barcode, slice_number = line.strip().split('\t')
                            if barcode in barcode_dict:
                                print(f"Processing barcode {barcode} (slice {slice_number})...")
                                # Fetch the set of permutations corresponding to the barcode from the dictionary
                                permutations = barcode_dict[barcode]
                                permut_search_set = set(permutations)
                            else:
                                raise RuntimeError(f"Barcode {barcode} not found in dictionary")
                            
                            # Create a temporary directory for this barcode.
                            # This is the output directory for the fastq_match_edit tool 
                            # This is where the dynamic files read their contents from.
                            dynamic_temp_dir = target_dir / "dynamic_temp_dir"
                            if dynamic_temp_dir.is_dir():
                                shutil.rmtree(dynamic_temp_dir)
                            dynamic_temp_dir.mkdir(parents=True, exist_ok=False)
                            
                            # Create a file with the barcode permutations for the program to read
                            permutation_file = Path(dynamic_temp_dir / f"{miss_prefix}_{barcode}_permutations.txt")
                            if permutation_file.is_file():
                                permutation_file.unlink()
                            with open(permutation_file, 'w') as f_out:
                                f_out.write(f"barcode_{barcode}_permutation" + '\n')
                                for permutation in permutations:
                                    f_out.write(permutation + '\n')
                            if not permutation_file.is_file() or not permutation_file.stat().st_size > 0:
                                raise RuntimeError(f"Failed to create permutation file {permutation_file}")
                            
                            # 1. Separate out the FASTQ records matching any of the permutations into temp output.
                            dynamic_prefix = f"barcode_{slice_number}_{barcode}"
                            seq_replace_command = ['python', fastq_match_edit,
                                                '--input-fastq', dynamic_miss_file,
                                                '--output-prefix', dynamic_prefix,
                                                '--output-dir', dynamic_temp_dir,
                                                '--seq-replace',
                                                '--query-seq-file', permutation_file,
                                                '--replacement-seq', barcode,
                                                '--log-dir', sample_log_dir,
                                                '--header-start', '@NS',
                                                '--processes', str(4) 
                                                ]
                            exit_status = execute_command(seq_replace_command)
                            if exit_status != 0:
                                raise subprocess.CalledProcessError(exit_status, seq_replace_command)
                            
                            # --- Update the dynamic files ---
                            # --------------------------------
                            corrected_output_list = list(dynamic_temp_dir.glob("*seq_replace.fastq"))
                            if not corrected_output_list:
                                raise RuntimeError(f"No corrected file found in {dynamic_temp_dir.name}")
                            elif len(corrected_output_list) > 1:
                                raise RuntimeError(f"Multiple corrected files found in {dynamic_temp_dir.name}: {corrected_output_list}")
                            else:
                                corrected_output_file = Path(corrected_output_list[0])
                            
                            # Update the corrected dynamic file
                            print(f"Appending contents: {corrected_output_file.name} --> {dynamic_corrected_file.name}")
                            with open(dynamic_corrected_file, 'ab') as f_out:
                                with open(corrected_output_file, 'rb') as f_in:
                                    shutil.copyfileobj(f_in, f_out)
                            print("done")
                            
                            # Fetch the mismatch output file
                            miss_output_list = list(dynamic_temp_dir.glob("*seq_mismatch.fastq"))
                            if not miss_output_list:
                                raise RuntimeError(f"No mismatch file found in {dynamic_temp_dir.name}")
                            elif len(miss_output_list) > 1:
                                raise RuntimeError(f"Multiple mismatch files found in {dynamic_temp_dir.name}: {miss_output_list}")
                            else:
                                miss_output_file = Path(miss_output_list[0])
                            
                            # Replace the contents of the dynamic mismatch file
                            print(f"Replacing contents: {miss_output_file.name} --> {dynamic_miss_file.name}")
                            if dynamic_miss_file.is_file():
                                dynamic_miss_file.unlink()
                            with open(dynamic_miss_file, 'wb') as f_out:
                                with open(miss_output_file, 'rb') as f_in:
                                    shutil.copyfileobj(f_in, f_out)
                            print('done')
                            
                            # Clean up the temp directory
                            if dynamic_temp_dir.is_dir():
                                shutil.rmtree(dynamic_temp_dir)
                finally:
                    # Clean up
                    if dynamic_temp_dir and dynamic_temp_dir.is_dir():
                        shutil.rmtree(dynamic_temp_dir)
                    if permutation_file and permutation_file.is_file():
                        permutation_file.unlink()
            finally:
                # --- End of the mismatch file loop ---
                # Finalize the correction
                if dynamic_corrected_file and dynamic_corrected_file.is_file():
                    print(f"Finalizing results for {miss_prefix}")
                    corrected_header_count = count_fq_headers(dynamic_corrected_file)
                    # Append the reads to the intact barcodes file
                    if intact_barcode_file and intact_barcode_file.is_file():
                        intact_header_count = count_fq_headers(intact_barcode_file)
                        with open(intact_barcode_file, 'ab') as f_out:
                            with open(dynamic_corrected_file, 'rb') as f_in:
                                shutil.copyfileobj(f_in, f_out)
                        barcode_logger.info(f"------- {miss_prefix} -------")
                        barcode_logger.info(f"Total barcode reads for {miss_prefix}: {intact_header_count:,} (intact) + {corrected_header_count:,} (corrected)")
                        barcode_logger.info(f"Appended corrected reads to {intact_barcode_file.name}")

                # Result summary for uncorrected barcodes
                if dynamic_miss_file and dynamic_miss_file.is_file():
                    miss_header_count = count_fq_headers(dynamic_miss_file)
                    barcode_logger.info(f"Total barcode reads left uncorrected for {miss_prefix}: {miss_header_count:,}")
                # Clean up
                if dynamic_corrected_file and dynamic_corrected_file.is_file():
                    dynamic_corrected_file.unlink()
                if dynamic_miss_file and dynamic_miss_file.is_file():
                    dynamic_miss_file.unlink()
finally:
    # Clean any temporary directories
    if barcode_correction_dir and barcode_correction_dir.is_dir():
        all_dir_paths = barcode_correction_dir.rglob("*")
        all_dir_paths = [x for x in all_dir_paths if x.is_dir()]
        for dir in all_dir_paths:
            if "_temp_" in dir.name:
                print(f"Removing temporary directory: {dir.name}")
                shutil.rmtree(dir)
        
        # Remove the uncorrected barcode files
        mismatch_file_list = barcode_correction_dir.rglob("*seq_mismatch*")
        mismatch_file_list = [path for path in mismatch_file_list if validate_fastq(path)]
        if mismatch_file_list:
            for mismatch_file in mismatch_file_list:
                mismatch_file.unlink()
                print(f"Removed uncorrected barcode file: {mismatch_file.name}")
        
        # Ensure contents are maintained in compressed form
        management_success = manage_directory_content_compression(barcode_correction_dir)
        if not management_success:
            raise RuntimeError(f"Failed to manage compression in directory {str(barcode_correction_dir)}")
    

### Demultiplexing

#### R1 demultiplex
The error-corrected barcode R1 reads are first demultiplexed. The R2 demultiplexing uses the header IDs of the demultiplexed R1 files.

In [ ]:
# --- Set up input and output for R1 demultiplexing ---
# -----------------------------------------------------
# Define the input directory
processed_reads_dir = qc_reads_dir / 'further_processing'
corrected_barcode_dir = processed_reads_dir / "barcode_correction"
if not corrected_barcode_dir.is_dir():
    raise FileNotFoundError(f"Barcode correction directory not found: {corrected_barcode_dir.name}")
# Ensure directory contents are maintained in compressed form
management_success = manage_directory_content_compression(corrected_barcode_dir)
if not management_success:
    raise RuntimeError(f"Failed to manage compression in directory {str(corrected_barcode_dir)}")

# Set up the output directory for demultiplexed reads
processed_reads_dir = qc_reads_dir / 'further_processing'
demultiplex_dir = Path(processed_reads_dir) / "demultiplexed_reads"
demultiplex_dir.mkdir(parents=True, exist_ok=True)
# logging
demultiplex_log_dir = log_dir / 'demultiplex_logs'
r1_demultiplex_log_dir = demultiplex_log_dir / f"R1_demultiplex_logs"
r1_demultiplex_log_dir.mkdir(parents=True, exist_ok=True)

# --- Iterate through the directories in the barcode correction directory ---
# ---------------------------------------------------------------------------
# It is assumed that a directory corresponds to a single sample, and that a directory contains 
# the *seq_match* R1 files in the sample directories contain the intact + error-corrected barcodes 
target_dir_list = corrected_barcode_dir.glob("*")
target_dir_list = [path for path in target_dir_list if path.is_dir()]
if not target_dir_list:
    raise FileNotFoundError(f"No target directories found in {corrected_barcode_dir.name}")
print(f"Found {len(target_dir_list)} target directories in {corrected_barcode_dir.name}")

for target_dir in target_dir_list:
    try:
        print(f"--- Processing directory: {target_dir.name} ---")

        # --- Fetch the input R1 file ---
        r1_file_list = list(target_dir.glob("*seq_match*"))
        r1_file_list = [path for path in r1_file_list if validate_fastq(path)]
        if not r1_file_list:
            raise FileNotFoundError(f"No R1 FASTQ files found in {target_dir}")
        elif len(r1_file_list) > 1:
            raise FileNotFoundError(f"Multiple R1 FASTQ files found in {target_dir}: {r1_file_list}")
        else:
            r1_file = r1_file_list[0]
        print(f"input file: {r1_file.name}")

        # Create a sample output directory
        sample_output_dir = Path(demultiplex_dir) / f"{target_dir.name}_demultiplexed" / f"R1_{target_dir.name}_demultiplexed"
        sample_output_dir.mkdir(parents=True, exist_ok=True)
        print(f"output directory: {format_short_path(sample_output_dir)}")

        # --- Run the R1 demultiplexer ---
        demultiplex_command = ["python", 
                        r1_demultiplex_program, 
                        "--r1-file", r1_file, 
                        "--barcode-file", barcode_list_file, 
                        "--dmplex-output-dir", sample_output_dir,
                        "--log-dir", r1_demultiplex_log_dir,
                        ]
        exit_status = execute_command(demultiplex_command)
        if exit_status != 0:
            raise subprocess.CalledProcessError(exit_status, demultiplex_command)
    finally:
        # Ensure directory contents are maintained in compressed form
        if target_dir and target_dir.is_dir():
            management_success = manage_directory_content_compression(target_dir)
            if not management_success:
                raise RuntimeError(f"Failed to manage compression in directory {str(target_dir)}")
        if sample_output_dir and sample_output_dir.is_dir():
            management_success = manage_directory_content_compression(sample_output_dir)
            if not management_success:
                raise RuntimeError(f"Failed to manage compression in directory {str(sample_output_dir)}")
    

#### R2 demultiplex

In [ ]:
# --- Set up input and output for R2 demultiplexing ---
# -----------------------------------------------------
# Ensure directory contents are maintained in compressed form
management_success = manage_directory_content_compression(qc_reads_dir)
if not management_success:
    raise RuntimeError(f"Failed to manage compression in directory {str(corrected_barcode_dir)}")

# Define the demultiplexed reads directory. This has to contain the demultiplexed R1 reads.
processed_reads_dir = Path(qc_reads_dir) / "further_processing"
demultiplex_dir = Path(processed_reads_dir) / "demultiplexed_reads"
if not demultiplex_dir.is_dir():
    raise FileNotFoundError(f"Demultiplexed reads directory not found: {demultiplex_dir}")
# logging
demultiplex_log_dir = log_dir / 'demultiplex_logs'
r2_demultiplex_log_dir = demultiplex_log_dir / f"R2_demultiplex_logs"
r2_demultiplex_log_dir.mkdir(parents=True, exist_ok=True)

# Fetch the R1 directories that contain the demultiplexed R1 reads
r1_dir_list = demultiplex_dir.rglob("R1_*_demultiplexed")
r1_dir_list = [path for path in r1_dir_list if path.is_dir() and path.name.startswith('R1_')]
if not r1_dir_list:
    raise FileNotFoundError(f"No target directories found in {demultiplex_dir.name}")
print(f"Found {len(r1_dir_list)} target directories in {demultiplex_dir.name}")

# Fetch the R2 directories that contain the R2 files to be demultiplexed
r2_dir_list = qc_reads_dir.glob("*")
r2_dir_list = [path for path in r2_dir_list if path.is_dir() and path != processed_reads_dir]
if not r2_dir_list:
    raise FileNotFoundError(f"No directories found in {qc_reads_dir.name}")

# --- Iterate through the R2 directories ---
# ------------------------------------------
# It is assumed that a directory corresponds to a single sample,
# R2 input file naming format: 'R2_<sample_name>.<extension>'
# R1 input directory naming format: 'R1_<sample_name>_demultiplexed'
for r2_dir in r2_dir_list:
    try:
        # --- Fetch the R2 input file ---
        # -------------------------------
        r2_file_list = list(r2_dir.glob("*R2_*"))
        r2_file_list = [path for path in r2_file_list if validate_fastq(path)]
        if not r2_file_list:
            raise FileNotFoundError(f"No R2 FASTQ files found in {r2_dir}")
        elif len(r2_file_list) > 1:
            raise FileNotFoundError(f"Multiple R2 FASTQ files found in {r2_dir}: {r2_file_list}")
        else:
            r2_input_file = r2_file_list[0]
        print(f"input file: {r2_input_file.name}")
        # Define sample ID from the input file name
        r2_sample_id = r2_input_file.name.split(".")[0].split("_")[1]

        # --- Fetch the R1 input directory ---
        # ------------------------------------
        r1_input_dir_list = []
        for r1_dir in r1_dir_list:
            r1_sample_id = r1_dir.name.split("_")[1]
            if r1_sample_id == r2_sample_id:
                r1_input_dir = r1_dir
                r1_input_dir_list.append(r1_input_dir)
        if not r1_input_dir_list:
            raise FileNotFoundError(f"No R1 input directory found for sample {r2_sample_id}")
        elif len(r1_input_dir_list) > 1:
            raise FileNotFoundError(f"Multiple R1 input directories found for sample {r2_sample_id}: {r1_input_dir_list}")
        else:
            r1_input_dir = r1_input_dir_list[0]
        print(f"R1 input directory: {format_short_path(r1_input_dir)}")

        # Create a sample output directory
        sample_output_dir = Path(demultiplex_dir) / f"{r2_sample_id}_demultiplexed" / f"R2_{r2_sample_id}_demultiplexed"
        sample_output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Output directory: {format_short_path(sample_output_dir)}")

        # --- Run the R2 demultiplexer ---
        # --------------------------------
        demultiplex_command = [
            "python", r2_demultiplex_program,
            '--r1-dmplex-dir', r1_input_dir,
            '--r2-file', r2_input_file,
            '--barcode-file', barcode_list_file,
            '--dmplex-output-dir', sample_output_dir,
            '--log-dir', r2_demultiplex_log_dir,
            '--processes', '4']
        
        exit_status = execute_command(demultiplex_command)
        if exit_status != 0:
            raise subprocess.CalledProcessError(exit_status, demultiplex_command)
    finally:
        # Ensure directory contents are maintained in compressed form
        if r2_dir and r2_dir.is_dir():
            management_success = manage_directory_content_compression(r2_dir)
            if not management_success:
                raise RuntimeError(f"Failed to manage compression in directory {str(r2_dir)}")
        if sample_output_dir and sample_output_dir.is_dir():
            management_success = manage_directory_content_compression(sample_output_dir)
            if not management_success:
                raise RuntimeError(f"Failed to manage compression in directory {str(sample_output_dir)}")

#### Demultiplexed read count summary
Summarize the read count in the demultiplexed files of each sample. 

In [ ]:
temp_file_list = qc_reads_dir.rglob('*')
temp_file_list = [path for path in temp_file_list if path.is_file() and path.name.startswith('tmp')]
if temp_file_list:
    print(f'Deleting {len(temp_file_list)} temporary files')
    for file in temp_file_list:
        file.unlink()
        print(f'Deleted {file.name}')

# Fetch the demultiplexed sample directory paths.
# It is assumed that each directory name contains a unique sample name.
qc_r2_files_list = qc_reads_dir.rglob('*R2*.qc.fastq*')
qc_r2_files_list = [path for path in qc_r2_files_list if path.is_file()]
processed_reads_dir = Path(qc_reads_dir) / "further_processing"
demultiplex_dir = Path(processed_reads_dir) / 'demultiplexed_reads'
demultiplex_dir_list = demultiplex_dir.glob('*')
demultiplex_dir_list = [path for path in demultiplex_dir_list if path.is_dir()]
if not demultiplex_dir_list:
    raise FileNotFoundError(f"No demultiplexed directories found in {demultiplex_dir}")

demultiplex_log_dir = log_dir / 'demultiplex_logs'
demultiplex_summary_log_dir = Path(demultiplex_log_dir) / 'demultiplex_summary_logs'
demultiplex_summary_log_dir.mkdir(parents=True, exist_ok=True)

# Iterate through the directories
for sample_dir in demultiplex_dir_list:
    sample_id = sample_dir.name.split('_')[0]
    print(f"Processing sample {sample_id}")

    # Fetch the not-demultiplexed QC R2 file
    parent_r2_file_list = []
    parent_r2_file = None
    for r2_file in qc_r2_files_list:
        r2_sample_id = r2_file.name.split('.')[0]
        r2_sample_id = r2_sample_id.split('_')[1]
        if r2_sample_id == sample_id:
            parent_r2_file_list.append(r2_file)
    if parent_r2_file_list:
        if len(parent_r2_file_list) > 1:
            raise ValueError("Multiple parent R2 files found")
        print(f"Found {len(parent_r2_file_list)} parent R2 file(s): {parent_r2_file_list}")
        parent_r2_file = parent_r2_file_list[0]
    if not parent_r2_file:
        raise FileNotFoundError("No parent R2 files found")

    # Fetch the demultiplexed R2 file paths from the sample dir
    sample_demultiplexed_path_list = sample_dir.rglob('*_R2_*.dmplx.*')
    sample_demultiplexed_path_list = [path for path in sample_demultiplexed_path_list if path.is_file()]
    if not sample_demultiplexed_path_list:
        raise FileNotFoundError(f"No sample demultiplexed files found for sample {sample_id}")
    print(f"Found {len(sample_demultiplexed_path_list)} files for sample {sample_id}")

    # Format the log output
    sample_log_file = Path(demultiplex_summary_log_dir) / f"{sample_id}_demultiplex_summary.log"

    # --- Step 1: Collect results into a dictionary ---
    sample_results = {}
    print(f"Counting reads for sample {sample_id}...")
    for file in sample_demultiplexed_path_list:
        try:
            barcode, barcode_id = barcode_info_from_filename(file.name)
            barcode_read_count = count_fq_headers(file) # Call your counting function
            # Using your desired key format
            barcode_key = f"{barcode} ({barcode_id})"
            # Store result - handle potential duplicate keys if needed (e.g., add counts)
            sample_results[barcode_key] = sample_results.get(barcode_key, 0) + barcode_read_count
        except Exception as e:
            print(f"  Error processing file {file.name}: {e}")
            # Decide how to handle: skip file, record error, etc.

    # --- Step 2: Sort the collected results by read count (value) ---
    # Sorts descending (highest count first). Remove reverse=True for ascending.
    try:
        sorted_results = sorted(
            sample_results.items(),
            key=lambda item: item[1], # Sort by the second element (the count)
            reverse=True
        )
    except Exception as e:
        print(f"Error sorting results for sample {sample_id}: {e}")
        continue # Skip writing for this sample if sorting fails

    # --- Step 3: Write the sorted and formatted results to the log file ---
    sample_log_file = Path(demultiplex_summary_log_dir) / f"{sample_id}_demultiplex_summary.log"
    print(f"Writing sorted summary log to {sample_log_file}")
    try:
        with open(sample_log_file, 'w') as f_out:
            # Write header - Ensure newline at the end!
            f_out.write("Barcode\tRead_count\n")

            # Iterate through the *sorted* list of (key, count) tuples
            for barcode_key, read_count in sorted_results:
                # Apply comma formatting to read_count using f-string specifier ':' followed by ','
                # Add newline character '\n'
                f_out.write(f"{barcode_key}\t{read_count:,}\n")
            
            parent_header_count = count_fq_headers(parent_r2_file)
            total_demultiplexed = sum(sample_results.values())
            f_out.write('\nSummary:\n')
            f_out.write(f"{parent_r2_file.name}: {parent_header_count:,}\n")
            dmplx_percent = round((total_demultiplexed / parent_header_count) * 100, 1)
            f_out.write(f"Total demultiplexed: {total_demultiplexed:,} ({dmplx_percent}%)")

    except IOError as e:
        print(f"  Error writing log file {sample_log_file}: {e}")
    except Exception as e:
        print(f"  An unexpected error occurred during writing for sample {sample_id}: {e}")


print("\nProcessing complete.")